In [1]:
# lyapunov_drift_check.py
# JAX sanity-check for: L Vbar <= -lambda Vbar + b
# where Vbar = 1 + mean_p (1 - (1/N) ReTr(U_p)) on a periodic L^4 lattice.

import jax
import jax.numpy as jnp
import jax.scipy.linalg as jsp

def haar_suN(key, shape, N):
    # Approx Haar via QR of complex Gaussian; adjust det to 1
    k1, k2 = jax.random.split(key)
    z = (jax.random.normal(k1, shape + (N, N)) +
         1j * jax.random.normal(k2, shape + (N, N))) / jnp.sqrt(2.0)
    q, r = jnp.linalg.qr(z)
    # Make diag(r) have positive real phase
    phase = jnp.exp(-1j * jnp.angle(jnp.diagonal(r, axis1=-2, axis2=-1)))
    q = q * phase[..., None, :]
    # Fix determinant to 1
    detq = jnp.linalg.det(q)
    q = q / detq[..., None, None] ** (1.0 / N)
    return q

def suN_tangent_gaussian(key, shape, N):
    # Gaussian in su(N): anti-Hermitian traceless
    k1, k2 = jax.random.split(key)
    a = (jax.random.normal(k1, shape + (N, N)) +
         1j * jax.random.normal(k2, shape + (N, N))) / jnp.sqrt(2.0)
    x = a - jnp.conjugate(jnp.swapaxes(a, -1, -2))  # anti-Hermitian
    tr = jnp.trace(x, axis1=-2, axis2=-1) / N
    x = x - tr[..., None, None] * jnp.eye(N, dtype=x.dtype)
    return x

def shift4(arr, axis, s):
    return jnp.roll(arr, shift=s, axis=axis)

def plaquette(U, mu, nu):
    # U: [L,L,L,L,4,N,N] periodic; mu,nu in {0,1,2,3}, mu<nu
    U_mu = U[..., mu, :, :]
    U_nu = U[..., nu, :, :]
    U_mu_x_nu = shift4(U_mu, axis=nu, s=1)
    U_nu_x_mu = shift4(U_nu, axis=mu, s=1)
    U_mu_dag_x_nu = jnp.conjugate(jnp.swapaxes(U_mu_x_nu, -1, -2))
    U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
    return U_mu @ U_nu_x_mu @ U_mu_dag_x_nu @ U_nu_dag

def Vbar(U):
    # Vbar = 1 + mean_{plaquettes} (1 - (1/N) ReTr U_p)
    N = U.shape[-1]
    z_list = []
    for mu in range(4):
        for nu in range(mu + 1, 4):
            Up = plaquette(U, mu, nu)
            tr = jnp.real(jnp.trace(Up, axis1=-2, axis2=-1))
            z = 1.0 - (1.0 / N) * tr
            z_list.append(z)
    z_all = jnp.stack(z_list, axis=0)  # [6,L,L,L,L]
    return 1.0 + jnp.mean(z_all)

def estimate_LV(U, beta, eps, mc_samples, key):
    # Stochastic estimate of L f = Delta f - <grad S, grad f>
    # using random Gaussian directions in su(N) on each link.
    L = U.shape[0]
    N = U.shape[-1]

    def S(U):
        # Wilson action (up to constant): beta * sum_p z_p
        z_sum = 0.0
        for mu in range(4):
            for nu in range(mu + 1, 4):
                Up = plaquette(U, mu, nu)
                tr = jnp.real(jnp.trace(Up, axis1=-2, axis2=-1))
                z = 1.0 - (1.0 / N) * tr
                z_sum = z_sum + jnp.sum(z)
        return beta * z_sum

    f = Vbar

    def one_sample(k):
        # independent tangent per link
        Xi = suN_tangent_gaussian(k, (L, L, L, L, 4), N)
        # Right-multiply each link by exp(±eps Xi)
        exp_p = jsp.expm(eps * Xi)
        exp_m = jsp.expm(-eps * Xi)
        U_p = U @ exp_p
        U_m = U @ exp_m

        f_p, f_0, f_m = f(U_p), f(U), f(U_m)
        S_p, S_0, S_m = S(U_p), S(U), S(U_m)

        # Laplacian estimator: E[(f(Ue^{eξ}) + f(Ue^{-eξ}) - 2f(U))/e^2]
        lap = (f_p + f_m - 2.0 * f_0) / (eps ** 2)

        # grad inner product estimator: E[(Dξ S)(Dξ f)]
        dS = (S_p - S_m) / (2.0 * eps)
        df = (f_p - f_m) / (2.0 * eps)
        gip = dS * df

        return lap - gip

    keys = jax.random.split(key, mc_samples)
    vals = jax.vmap(one_sample)(keys)
    return jnp.mean(vals), jnp.std(vals) / jnp.sqrt(mc_samples)

def main():
    key = jax.random.PRNGKey(0)
    N = 3
    L = 2  # keep small for sanity checks
    beta = 6.0
    eps = 5e-3
    mc = 128

    key, kU, kL = jax.random.split(key, 3)
    U = haar_suN(kU, (L, L, L, L, 4), N)

    v = Vbar(U)
    LV_est, LV_se = estimate_LV(U, beta=beta, eps=eps, mc_samples=mc, key=kL)

    # Choose a Laplacian normalization:
    # If your Laplacian matches the fundamental Casimir C_F, set:
    # lambda = 4*C_F, b = 8*C_F.
    # For a quick numeric check, treat C_F= (N^2-1)/(2N) in the common normalization.
    C_F = (N**2 - 1) / (2.0 * N)
    lam = 4.0 * C_F
    b = 8.0 * C_F

    rhs = -lam * v + b

    print("Vbar(U) =", float(v))
    print("Estimated L Vbar =", float(LV_est), "+/-", float(LV_se))
    print("RHS = -lambda Vbar + b =", float(rhs))
    print("Check: LV <= RHS ?", bool(LV_est <= rhs + 5.0 * LV_se))

if __name__ == "__main__":
    main()

Vbar(U) = 2.0038504600524902
Estimated L Vbar = -8.81934928894043 +/- 1.0230703353881836
RHS = -lambda Vbar + b = -0.02053546905517578
Check: LV <= RHS ? True


In [3]:
# drift_batch_check_fixed2.py
import jax
import jax.numpy as jnp
import jax.scipy.linalg as jsp
from functools import partial

def haar_suN(key, shape, N):
    k1, k2 = jax.random.split(key)
    z = (jax.random.normal(k1, shape + (N, N)) +
         1j * jax.random.normal(k2, shape + (N, N))) / jnp.sqrt(2.0)
    q, r = jnp.linalg.qr(z)
    phase = jnp.exp(-1j * jnp.angle(jnp.diagonal(r, axis1=-2, axis2=-1)))
    q = q * phase[..., None, :]
    detq = jnp.linalg.det(q)
    q = q / detq[..., None, None] ** (1.0 / N)
    return q

def suN_tangent_gaussian(key, shape, N):
    k1, k2 = jax.random.split(key)
    a = (jax.random.normal(k1, shape + (N, N)) +
         1j * jax.random.normal(k2, shape + (N, N))) / jnp.sqrt(2.0)
    x = a - jnp.conjugate(jnp.swapaxes(a, -1, -2))  # anti-Hermitian
    tr = jnp.trace(x, axis1=-2, axis2=-1) / N
    x = x - tr[..., None, None] * jnp.eye(N, dtype=x.dtype)  # traceless
    return x

def shift4(arr, axis, s):
    return jnp.roll(arr, shift=s, axis=axis)

def plaquette(U, mu, nu):
    U_mu = U[..., mu, :, :]
    U_nu = U[..., nu, :, :]
    U_mu_x_nu = shift4(U_mu, axis=nu, s=1)
    U_nu_x_mu = shift4(U_nu, axis=mu, s=1)
    U_mu_dag_x_nu = jnp.conjugate(jnp.swapaxes(U_mu_x_nu, -1, -2))
    U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
    return U_mu @ U_nu_x_mu @ U_mu_dag_x_nu @ U_nu_dag

def z_stack(U):
    N = U.shape[-1]
    z_list = []
    for mu in range(4):
        for nu in range(mu + 1, 4):
            Up = plaquette(U, mu, nu)
            tr = jnp.real(jnp.trace(Up, axis1=-2, axis2=-1))
            z = 1.0 - (1.0 / N) * tr
            z_list.append(z)
    return jnp.stack(z_list, axis=0)  # [6, L,L,L,L]

def zsum_and_Vbar(U):
    z_all = z_stack(U)
    z_sum = jnp.sum(z_all)
    V = 1.0 + jnp.mean(z_all)
    return z_sum, V

@partial(jax.jit, static_argnums=(4,5,6))
def estimate_LV_one(U, beta, eps, key, mc_samples: int, N: int, L: int):
    # Works for any beta, including beta=0 (then drift term is identically 0).
    def one_dir(k):
        Xi = suN_tangent_gaussian(k, (L, L, L, L, 4), N)
        exp_p = jsp.expm(eps * Xi)
        exp_m = jsp.expm(-eps * Xi)

        U_p = U @ exp_p
        U_m = U @ exp_m

        zsum_p, V_p = zsum_and_Vbar(U_p)
        zsum_0, V_0 = zsum_and_Vbar(U)
        zsum_m, V_m = zsum_and_Vbar(U_m)

        lap = (V_p + V_m - 2.0 * V_0) / (eps ** 2)

        # Wilson action difference (beta may be 0; then dS=0)
        S_p = beta * zsum_p
        S_m = beta * zsum_m
        dS = (S_p - S_m) / (2.0 * eps)

        dV = (V_p - V_m) / (2.0 * eps)
        return lap - dS * dV

    keys = jax.random.split(key, mc_samples)
    vals = jax.vmap(one_dir)(keys)
    return jnp.mean(vals), jnp.std(vals) / jnp.sqrt(mc_samples)

@partial(jax.jit, static_argnums=(4,5,6))
def run_batch(keysU, keysL, beta, eps, mc_samples: int, N: int, L: int):
    def one(kU, kL):
        U = haar_suN(kU, (L, L, L, L, 4), N)
        _, v = zsum_and_Vbar(U)
        lv, se = estimate_LV_one(U, beta=beta, eps=eps, key=kL,
                                 mc_samples=mc_samples, N=N, L=L)
        return v, lv, se
    return jax.vmap(one)(keysU, keysL)

if __name__ == "__main__":
    N = 3
    L = 2

    eps = 5e-3
    mc = 256
    K = 64

    key = jax.random.PRNGKey(0)
    key, kU, kL = jax.random.split(key, 3)
    keysU = jax.random.split(kU, K)
    keysL = jax.random.split(kL, K)

    C_F = (N**2 - 1) / (2.0 * N)
    lam = 4.0 * C_F
    b = 8.0 * C_F

    # 1) Laplacian normalization check: beta = 0
    beta0 = 0.0
    v0, lv0, se0 = run_batch(keysU, keysL, beta0, eps, mc, N, L)
    rhs0 = -lam * v0 + b
    err0 = lv0 - rhs0

    print("=== beta=0 Laplacian identity check ===")
    print("mean(V)          =", float(jnp.mean(v0)))
    print("mean(Delta V)    =", float(jnp.mean(lv0)))
    print("mean(RHS)        =", float(jnp.mean(rhs0)))
    print("mean(DeltaV-RHS) =", float(jnp.mean(err0)))
    print("max |DeltaV-RHS| =", float(jnp.max(jnp.abs(err0))))
    print("mean SE(Delta V) =", float(jnp.mean(se0)))

    # 2) Full generator check: beta > 0
    beta = 6.0
    v, lv, se = run_batch(keysU, keysL, beta, eps, mc, N, L)
    rhs = -lam * v + b
    slack = rhs - lv

    print("\n=== beta>0 drift inequality check ===")
    print("mean(V)            =", float(jnp.mean(v)))
    print("mean(L V)          =", float(jnp.mean(lv)))
    print("mean(RHS)          =", float(jnp.mean(rhs)))
    print("min slack (rhs-lv) =", float(jnp.min(slack)))
    print("mean slack         =", float(jnp.mean(slack)))
    print("mean SE(L V)       =", float(jnp.mean(se)))

=== beta=0 Laplacian identity check ===
mean(V)          = 2.001002788543701
mean(Delta V)    = -0.021118907257914543
mean(RHS)        = -0.005349263548851013
mean(DeltaV-RHS) = -0.01576964743435383
max |DeltaV-RHS| = 1.3196303844451904
mean SE(Delta V) = 0.0523807629942894

=== beta>0 drift inequality check ===
mean(V)            = 2.001002788543701
mean(L V)          = -7.058921813964844
mean(RHS)          = -0.005349263548851013
min slack (rhs-lv) = 4.528066635131836
mean slack         = 7.053572654724121
mean SE(L V)       = 0.6091587543487549


In [4]:
# drift_batch_check_fixed3.py
import jax
import jax.numpy as jnp
import jax.scipy.linalg as jsp
from functools import partial

def haar_suN(key, shape, N):
    k1, k2 = jax.random.split(key)
    z = (jax.random.normal(k1, shape + (N, N)) +
         1j * jax.random.normal(k2, shape + (N, N))) / jnp.sqrt(2.0)
    q, r = jnp.linalg.qr(z)
    phase = jnp.exp(-1j * jnp.angle(jnp.diagonal(r, axis1=-2, axis2=-1)))
    q = q * phase[..., None, :]
    detq = jnp.linalg.det(q)
    q = q / detq[..., None, None] ** (1.0 / N)
    return q

def suN_tangent_gaussian(key, shape, N):
    k1, k2 = jax.random.split(key)
    a = (jax.random.normal(k1, shape + (N, N)) +
         1j * jax.random.normal(k2, shape + (N, N))) / jnp.sqrt(2.0)
    x = a - jnp.conjugate(jnp.swapaxes(a, -1, -2))  # anti-Hermitian
    tr = jnp.trace(x, axis1=-2, axis2=-1) / N
    x = x - tr[..., None, None] * jnp.eye(N, dtype=x.dtype)  # traceless
    return x

def suN_norm2(X):
    # ||X||^2 = -ReTr(X^2) (X anti-Hermitian => Tr(X^2) real <= 0)
    return -jnp.real(jnp.trace(X @ X, axis1=-2, axis2=-1))

def shift4(arr, axis, s):
    return jnp.roll(arr, shift=s, axis=axis)

def plaquette(U, mu, nu):
    U_mu = U[..., mu, :, :]
    U_nu = U[..., nu, :, :]
    U_mu_x_nu = shift4(U_mu, axis=nu, s=1)
    U_nu_x_mu = shift4(U_nu, axis=mu, s=1)
    U_mu_dag_x_nu = jnp.conjugate(jnp.swapaxes(U_mu_x_nu, -1, -2))
    U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
    return U_mu @ U_nu_x_mu @ U_mu_dag_x_nu @ U_nu_dag

def z_stack(U):
    N = U.shape[-1]
    z_list = []
    for mu in range(4):
        for nu in range(mu + 1, 4):
            Up = plaquette(U, mu, nu)
            tr = jnp.real(jnp.trace(Up, axis1=-2, axis2=-1))
            z = 1.0 - (1.0 / N) * tr
            z_list.append(z)
    return jnp.stack(z_list, axis=0)  # [6, L,L,L,L]

def zsum_and_Vbar(U):
    z_all = z_stack(U)
    z_sum = jnp.sum(z_all)
    V = 1.0 + jnp.mean(z_all)
    return z_sum, V

@partial(jax.jit, static_argnums=(5,6,7))
def estimate_LV_one(U, beta, eps, key, scale, mc_samples: int, N: int, L: int):
    # Finite-diff estimate of L Vbar with calibrated tangent scaling.
    def one_dir(k):
        Xi = scale * suN_tangent_gaussian(k, (L, L, L, L, 4), N)
        exp_p = jsp.expm(eps * Xi)
        exp_m = jsp.expm(-eps * Xi)

        U_p = U @ exp_p
        U_m = U @ exp_m

        zsum_p, V_p = zsum_and_Vbar(U_p)
        _,      V_0 = zsum_and_Vbar(U)
        zsum_m, V_m = zsum_and_Vbar(U_m)

        lap = (V_p + V_m - 2.0 * V_0) / (eps ** 2)

        # drift part (works even if beta=0; then dS=0)
        S_p = beta * zsum_p
        S_m = beta * zsum_m
        dS = (S_p - S_m) / (2.0 * eps)

        dV = (V_p - V_m) / (2.0 * eps)
        return lap - dS * dV

    keys = jax.random.split(key, mc_samples)
    vals = jax.vmap(one_dir)(keys)
    return jnp.mean(vals), jnp.std(vals) / jnp.sqrt(mc_samples)

@partial(jax.jit, static_argnums=(5,6,7))
def run_batch(keysU, keysL, beta, eps, scale, mc_samples: int, N: int, L: int):
    def one(kU, kL):
        U = haar_suN(kU, (L, L, L, L, 4), N)
        _, v = zsum_and_Vbar(U)
        lv, se = estimate_LV_one(U, beta=beta, eps=eps, key=kL, scale=scale,
                                 mc_samples=mc_samples, N=N, L=L)
        return v, lv, se
    return jax.vmap(one)(keysU, keysL)

def calibrate_scale(key, N, M=20000):
    # Calibrate so E||X||^2 = dimG in the ||X||^2 = -ReTr(X^2) metric.
    dimG = N**2 - 1
    keys = jax.random.split(key, M)
    X = jax.vmap(lambda k: suN_tangent_gaussian(k, (), N))(keys)  # [M,N,N]
    mean_norm2 = jnp.mean(suN_norm2(X))
    scale = jnp.sqrt(dimG / mean_norm2)
    return float(scale), float(mean_norm2)

if __name__ == "__main__":
    N = 3
    L = 2

    eps = 5e-3
    mc = 512      # bump these for cleaner beta=0 identity check
    K  = 128

    key = jax.random.PRNGKey(0)
    key, kU, kL, kC = jax.random.split(key, 4)

    scale, mean_norm2 = calibrate_scale(kC, N, M=20000)
    print("Calibrated tangent scale =", scale, " (raw E||X||^2 =", mean_norm2, ")")

    keysU = jax.random.split(kU, K)
    keysL = jax.random.split(kL, K)

    # Casimir in the common normalization; your calibrated Laplacian should now match this scaling much better.
    C_F = (N**2 - 1) / (2.0 * N)
    lam = 4.0 * C_F
    b = 8.0 * C_F

    # beta=0 Laplacian identity check (your estimator returns Delta V now)
    beta0 = 0.0
    v0, dv0, se0 = run_batch(keysU, keysL, beta0, eps, scale, mc, N, L)
    rhs0 = -lam * v0 + b
    err0 = dv0 - rhs0

    print("\n=== beta=0 Laplacian identity check ===")
    print("mean(V)           =", float(jnp.mean(v0)))
    print("mean(Delta V)     =", float(jnp.mean(dv0)))
    print("mean(RHS)         =", float(jnp.mean(rhs0)))
    print("mean(DeltaV-RHS)  =", float(jnp.mean(err0)))
    print("max |DeltaV-RHS|  =", float(jnp.max(jnp.abs(err0))))
    print("mean SE(Delta V)  =", float(jnp.mean(se0)))

    # beta>0 drift inequality check
    beta = 6.0
    v, lv, se = run_batch(keysU, keysL, beta, eps, scale, mc, N, L)
    rhs = -lam * v + b
    slack = rhs - lv

    print("\n=== beta>0 drift inequality check ===")
    print("mean(V)             =", float(jnp.mean(v)))
    print("mean(L V)           =", float(jnp.mean(lv)))
    print("mean(RHS)           =", float(jnp.mean(rhs)))
    print("min slack (rhs-lv)  =", float(jnp.min(slack)))
    print("mean slack          =", float(jnp.mean(slack)))
    print("mean SE(L V)        =", float(jnp.mean(se)))

Calibrated tangent scale = 0.7055737376213074  (raw E||X||^2 = 16.069604873657227 )

=== beta=0 Laplacian identity check ===
mean(V)           = 2.0005240440368652
mean(Delta V)     = -0.014731924049556255
mean(RHS)         = -0.002794325351715088
mean(DeltaV-RHS)  = -0.011937598697841167
max |DeltaV-RHS|  = 1.3631060123443604
mean SE(Delta V)  = 0.02514338120818138

=== beta>0 drift inequality check ===
mean(V)             = 2.0005240440368652
mean(L V)           = -3.544910430908203
mean(RHS)           = -0.002794325351715088
min slack (rhs-lv)  = 1.9861762523651123
mean slack          = 3.542116165161133
mean SE(L V)        = 0.22066953778266907


In [6]:
# drift_batch_check_fixed4_basis_njit.py
import jax
import jax.numpy as jnp
import jax.scipy.linalg as jsp

def make_suN_orthonormal_basis(N: int, dtype=jnp.complex64):
    # Orthonormal for <A,B> = -ReTr(A B) on anti-Hermitian traceless matrices.
    eye = jnp.eye(N, dtype=dtype)

    def E(i, j):
        return eye[:, i:i+1] @ eye[j:j+1, :]

    basis = []

    # Off-diagonal generators (i<j)
    for i in range(N):
        for j in range(i + 1, N):
            A = E(i, j) - E(j, i)           # real antisymmetric (anti-Hermitian)
            B = 1j * (E(i, j) + E(j, i))    # imaginary symmetric (anti-Hermitian)
            basis.append(A)
            basis.append(B)

    # Diagonal (Cartan) generators
    for k in range(1, N):
        diag = jnp.concatenate([
            jnp.ones((k,), dtype=dtype),
            jnp.array([-k], dtype=dtype),
            jnp.zeros((N - k - 1,), dtype=dtype),
        ])
        H = 1j * jnp.diag(diag)
        basis.append(H)

    T = jnp.stack(basis, axis=0)  # [dimG, N, N]

    def norm2(X):
        return -jnp.real(jnp.trace(X @ X, axis1=-2, axis2=-1))

    n2 = jax.vmap(norm2)(T)
    T = T / jnp.sqrt(n2)[:, None, None]
    return T

def casimir_fundamental(Tbasis):
    # For anti-Hermitian orthonormal basis: sum_a T_a T_a = -C_F I
    S = jnp.einsum('aij,ajk->ik', Tbasis, Tbasis)  # [N,N]
    N = S.shape[0]
    return float(-jnp.real(jnp.trace(S)) / N)

def haar_suN(key, shape, N):
    k1, k2 = jax.random.split(key)
    z = (jax.random.normal(k1, shape + (N, N)) +
         1j * jax.random.normal(k2, shape + (N, N))) / jnp.sqrt(2.0)
    q, r = jnp.linalg.qr(z)
    phase = jnp.exp(-1j * jnp.angle(jnp.diagonal(r, axis1=-2, axis2=-1)))
    q = q * phase[..., None, :]
    detq = jnp.linalg.det(q)
    q = q / detq[..., None, None] ** (1.0 / N)
    return q

def shift4(arr, axis, s):
    return jnp.roll(arr, shift=s, axis=axis)

def plaquette(U, mu, nu):
    U_mu = U[..., mu, :, :]
    U_nu = U[..., nu, :, :]
    U_mu_x_nu = shift4(U_mu, axis=nu, s=1)
    U_nu_x_mu = shift4(U_nu, axis=mu, s=1)
    U_mu_dag_x_nu = jnp.conjugate(jnp.swapaxes(U_mu_x_nu, -1, -2))
    U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
    return U_mu @ U_nu_x_mu @ U_mu_dag_x_nu @ U_nu_dag

def z_stack(U):
    N = U.shape[-1]
    z_list = []
    for mu in range(4):
        for nu in range(mu + 1, 4):
            Up = plaquette(U, mu, nu)
            tr = jnp.real(jnp.trace(Up, axis1=-2, axis2=-1))
            z = 1.0 - (1.0 / N) * tr
            z_list.append(z)
    return jnp.stack(z_list, axis=0)  # [6, L,L,L,L]

def zsum_and_Vbar(U):
    z_all = z_stack(U)
    z_sum = jnp.sum(z_all)
    V = 1.0 + jnp.mean(z_all)
    return z_sum, V

def estimate_LV_one(U, beta, eps, key, Tbasis, mc_samples: int):
    # Finite-diff estimate of L Vbar (beta may be 0).
    L = U.shape[0]
    N = U.shape[-1]
    dimG = Tbasis.shape[0]

    def one_dir(k):
        coeff = jax.random.normal(k, (L, L, L, L, 4, dimG))
        Xi = jnp.einsum('.....a,aij->.....ij', coeff, Tbasis)  # [L,L,L,L,4,N,N]

        exp_p = jsp.expm(eps * Xi)
        exp_m = jsp.expm(-eps * Xi)

        U_p = U @ exp_p
        U_m = U @ exp_m

        zsum_p, V_p = zsum_and_Vbar(U_p)
        _,      V_0 = zsum_and_Vbar(U)
        zsum_m, V_m = zsum_and_Vbar(U_m)

        lap = (V_p + V_m - 2.0 * V_0) / (eps ** 2)

        S_p = beta * zsum_p
        S_m = beta * zsum_m
        dS = (S_p - S_m) / (2.0 * eps)
        dV = (V_p - V_m) / (2.0 * eps)

        return lap - dS * dV

    keys = jax.random.split(key, mc_samples)
    vals = jax.vmap(one_dir)(keys)
    return jnp.mean(vals), jnp.std(vals) / jnp.sqrt(mc_samples)

def run_batch(keysU, keysL, beta, eps, Tbasis, mc_samples: int):
    def one(kU, kL):
        # L is closed over from global scope
        U = haar_suN(kU, (L, L, L, L, 4), N)
        _, v = zsum_and_Vbar(U)
        lv, se = estimate_LV_one(U, beta, eps, kL, Tbasis, mc_samples)
        return v, lv, se
    return jax.vmap(one)(keysU, keysL)

if __name__ == "__main__":
    N = 3
    L = 2

    eps = 2e-3
    mc = 512
    K  = 128

    Tbasis = make_suN_orthonormal_basis(N)
    C_F = casimir_fundamental(Tbasis)
    lam = 4.0 * C_F
    b   = 8.0 * C_F
    print("Computed C_F =", C_F, "  lambda =", lam, "  b =", b)

    key = jax.random.PRNGKey(0)
    key, kU, kL = jax.random.split(key, 3)
    keysU = jax.random.split(kU, K)
    keysL = jax.random.split(kL, K)

    # beta=0 Laplacian identity check
    beta0 = 0.0
    v0, dv0, se0 = run_batch(keysU, keysL, beta0, eps, Tbasis, mc)
    rhs0 = -lam * v0 + b
    err0 = dv0 - rhs0

    print("\n=== beta=0 Laplacian identity check ===")
    print("mean(V)           =", float(jnp.mean(v0)))
    print("mean(Delta V)     =", float(jnp.mean(dv0)))
    print("mean(RHS)         =", float(jnp.mean(rhs0)))
    print("mean(DeltaV-RHS)  =", float(jnp.mean(err0)))
    print("max |DeltaV-RHS|  =", float(jnp.max(jnp.abs(err0))))
    print("mean SE(Delta V)  =", float(jnp.mean(se0)))

    # beta>0 drift inequality check
    beta = 6.0
    v, lv, se = run_batch(keysU, keysL, beta, eps, Tbasis, mc)
    rhs = -lam * v + b
    slack = rhs - lv

    print("\n=== beta>0 drift inequality check ===")
    print("mean(V)             =", float(jnp.mean(v)))
    print("mean(L V)           =", float(jnp.mean(lv)))
    print("mean(RHS)           =", float(jnp.mean(rhs)))
    print("min slack (rhs-lv)  =", float(jnp.min(slack)))
    print("mean slack          =", float(jnp.mean(slack)))
    print("mean SE(L V)        =", float(jnp.mean(se)))

Computed C_F = 2.6660945415496826   lambda = 10.66437816619873   b = 21.32875633239746


ValueError: Invalid Ellipses.